### Topics:  
* BM25 (keyword search)
* FAISS (semantic search)
* Compare both
* Build your own Hybrid Retriever
* Remove duplicate documents
* Send best context to Gemini

In [8]:
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_community.retrievers import BM25Retriever

from google import genai

**Load PDFs**

In [9]:
folder = "day16_pdfs"

documents = []

for file in os.listdir (folder):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(folder, file))
        documents.extend(loader.load())

print("Pages Loade:", len(documents))

Pages Loade: 121


**Check metadata**

In [10]:
print(documents[0].metadata)

{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'day16_pdfs\\attension-is-all-u-need.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}


**Split documents into chunks**

In [11]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 863


**Create embeddings**

In [12]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

**Create FAISS vector databse**

In [13]:
vector_db = FAISS.from_documents(
    chunks,
    embedding=embeddings
)

print("Vector Database Created Successfully!")

Vector Database Created Successfully!


**Create BM25 Retriever**

In [16]:
!pip install rank-bm25

In [17]:
from rank_bm25 import BM25Okapi

print("rank_bm25 installed successfully!")

rank_bm25 installed successfully!


In [18]:
bm25 = BM25Retriever.from_documents(chunks)
bm25.k = 3
print("BM25 Retriever Created!")

BM25 Retriever Created!


**Create FAISS Retriever**

In [19]:
vector_retriever = vector_db.as_retriever(
    search_kwargs={"k":3}
)

**Test BM25**

In [20]:
query = "What is BERT?"

bm25_docs = bm25.invoke(query)

for doc in bm25_docs:
    print(doc.page_content)

Gemini: A Family of Highly Capable Multimodal Models
10.4.4. Image understanding and reasoning
Prompt
Look at this sequence of three shapes. What shape should come as the fourth shape? Explain
your reasoning with detailed descriptions of the first shapes.
Model Response
The fourth shape should be a hexagon. The first shape is a triangle, the second shape is a
square, and the third shape is a pentagon. The number of sides in each shape is increasing
by one
• (too Latin-script centric) What is a word with 1 E, 2 As, and 1 U?
Translation Quality Validation: Each translated prompt-response pair was rated for translation
quality by at least 3 human raters, and was kept in the final mixture if the majority of raters rated it
as accurate. Section 5.1.4 reports evaluations of the multilingual capabilities of post-trained Gemini
API models.
6.5.4. Multimodal Vision
response rating by humans.
Translatability Filtering: Not all prompt-response pairs make sense when automatically translated,
and m

**Test FAISS**

In [21]:
faiss_docs = vector_retriever.invoke(query)

for doc in faiss_docs:
    print(doc.page_content)

asTi∈ RH.
For a given token, its input representation is
constructed by summing the corresponding token,
segment, and position embeddings. A visualiza-
tion of this construction can be seen in Figure 2.
3.1 Pre-training BERT
Unlike Peters et al. (2018a) and Radford et al.
(2018), we do not use traditional left-to-right or
right-to-left language models to pre-train BERT.
Instead, we pre-train BERT using two unsuper-
vised tasks, described in this section. This step
rameters as BERTBASE :
No NSP : A bidirectional model which is trained
using the “masked LM” (MLM) but without the
“next sentence prediction” (NSP) task.
LTR & No NSP: A left-context-only model which
is trained using a standard Left-to-Right (LTR)
LM, rather than an MLM. The left-only constraint
was also applied at ﬁne-tuning, because removing
it introduced a pre-train/ﬁne-tune mismatch that
degraded downstream performance. Additionally,
this model was pre-trained without the NSP task.
GLUE leaderboard10, BERTLARGE obtains a 

**New Hybrid Retriever
Instead of EnsembleRetriever**

In [22]:
def hybrid_search(query):

    bm25_results = bm25.invoke(query)

    faiss_results = vector_retriever.invoke(query)

    combined = bm25_results + faiss_results

    unique_docs = []

    seen = set()

    for doc in combined:

        if doc.page_content not in seen:

            unique_docs.append(doc)

            seen.add(doc.page_content)

    return unique_docs[:4]

In [23]:
docs = hybrid_search("What is BERT?")

for d in docs:
    print(d.page_content)

Gemini: A Family of Highly Capable Multimodal Models
10.4.4. Image understanding and reasoning
Prompt
Look at this sequence of three shapes. What shape should come as the fourth shape? Explain
your reasoning with detailed descriptions of the first shapes.
Model Response
The fourth shape should be a hexagon. The first shape is a triangle, the second shape is a
square, and the third shape is a pentagon. The number of sides in each shape is increasing
by one
• (too Latin-script centric) What is a word with 1 E, 2 As, and 1 U?
Translation Quality Validation: Each translated prompt-response pair was rated for translation
quality by at least 3 human raters, and was kept in the final mixture if the majority of raters rated it
as accurate. Section 5.1.4 reports evaluations of the multilingual capabilities of post-trained Gemini
API models.
6.5.4. Multimodal Vision
response rating by humans.
Translatability Filtering: Not all prompt-response pairs make sense when automatically translated,
and m

**Create Context**

In [24]:
context = "\n\n".join(
    [doc.page_content for doc in docs]
)

**prompt**

In [26]:
query = "What is BERT?"

docs = hybrid_search(query)

context = "\n\n".join(
    [doc.page_content for doc in docs]
)

prompt = f"""
You are an AI assistant.

Answer ONLY using the context below.

If the answer is not available, reply:

"I don't know based on the provided context."

Keep the answer short and relevant.

Context:
{context}

Question:
{query}

Answer:
"""

print(prompt)


You are an AI assistant.

Answer ONLY using the context below.

If the answer is not available, reply:

"I don't know based on the provided context."

Keep the answer short and relevant.

Context:
Gemini: A Family of Highly Capable Multimodal Models
10.4.4. Image understanding and reasoning
Prompt
Look at this sequence of three shapes. What shape should come as the fourth shape? Explain
your reasoning with detailed descriptions of the first shapes.
Model Response
The fourth shape should be a hexagon. The first shape is a triangle, the second shape is a
square, and the third shape is a pentagon. The number of sides in each shape is increasing
by one

• (too Latin-script centric) What is a word with 1 E, 2 As, and 1 U?
Translation Quality Validation: Each translated prompt-response pair was rated for translation
quality by at least 3 human raters, and was kept in the final mixture if the majority of raters rated it
as accurate. Section 5.1.4 reports evaluations of the multilingual capab

**Send Prompt to Gemini**

In [27]:
from google import genai

client = genai.Client(api_key="Your_API_Key")

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print(response.text)

I don't know based on the provided context.


**Build a Complete Hybrid Search Chatbot**

In [31]:
def ask_question(question):

    docs = hybrid_search(question)

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    prompt = f"""
    You are an AI assistant.

Rules:
1. Answer ONLY using the provided context.
2. If the answer is not found in the context, reply exactly:
   "I don't know based on the provided context."
3. Keep the answer short and concise (maximum 2-3 sentences).
4. Do NOT copy long paragraphs from the context.
5. Answer only the user's question.
6. If the question asks "Who", return only the person's or organization's name.
7. If the question asks "What", provide only the definition or explanation.
8. If the question asks "When", provide only the date or time.
9. If the question asks "Where", provide only the location.
10. Do not include extra details unless the user specifically asks for them.

    Context:
    {context}

    Question:
    {question}

    Answer:
    """

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

In [33]:
print(ask_question("What is the Transformer architecture?"))

The Transformer follows an overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder.


**Final Chat Bot**

In [35]:
while True:

    question = input("You: ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    answer = ask_question(question)

    print("\nAssistant:")
    print(answer)
    print("-" * 60)

You:  describe the model architecture of transformer.



Assistant:
The Transformer model architecture is an encoder-decoder structure. It uses stacked self-attention and point-wise, fully connected layers for both the encoder and decoder. The encoder is composed of a stack of N=6 identical layers, each with a multi-head self-attention mechanism and a simple, position-wise fully connected layer.
------------------------------------------------------------


You:  exit


Goodbye!
